# Fine-tune Llama 3.2 1B for Recipe Generation

**Purpose**: Fine-tune Llama 3.2 1B using QLoRA for high-quality recipe generation

**Method**: QLoRA (Quantized Low-Rank Adaptation)
- Memory efficient: Can run on 6GB GPU
- Fast training: Only trains ~1% of parameters
- High quality: Minimal performance loss

**Requirements**:
- GPU with 6GB+ VRAM (RTX 3060 or better)
- ~10GB disk space for model
- Training data: Recipe dataset from project

## 1. Environment Setup

In [2]:
# Install required packages
!pip install --upgrade transformers==4.46.0 accelerate==1.2.1 peft==0.13.2 bitsandbytes==0.45.0 datasets==3.2.0 trl==0.12.2

print("✓ Packages installed")

Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com
✓ Packages installed


In [7]:
import os
import json
import torch
from pathlib import Path
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    pipeline
)
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model
from trl import SFTTrainer

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

PyTorch version: 2.7.1+cu118
CUDA available: True
GPU: NVIDIA GeForce RTX 3060 Laptop GPU
VRAM: 6.0 GB


## 2. Configuration

In [15]:
from huggingface_hub import login

login()

In [8]:
# Project paths
PROJECT_ROOT = Path.cwd().parent.parent
DATA_DIR = PROJECT_ROOT / "data"
MODEL_DIR = PROJECT_ROOT / "models" / "recipe_generation"
OUTPUT_DIR = MODEL_DIR / "llama3_1b_finetuned"

# Recipe data
RECIPE_DATA_FILE = DATA_DIR / "processed" / "recipes" / "full_recipes.json"

# Model configuration
MODEL_NAME = "meta-llama/Llama-3.2-1B-Instruct"  # HuggingFace model
# Note: You need HuggingFace access token for Llama models

# Create directories
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Recipe data: {RECIPE_DATA_FILE}")
print(f"Output directory: {OUTPUT_DIR}")

Project root: c:\Users\Champion\Documents\GitHub\cAIuldron
Recipe data: c:\Users\Champion\Documents\GitHub\cAIuldron\data\processed\recipes\full_recipes.json
Output directory: c:\Users\Champion\Documents\GitHub\cAIuldron\models\recipe_generation\llama3_1b_finetuned


## 3. Load Training Data

In [9]:
# Load recipe dataset
print("Loading recipe data...")

if not RECIPE_DATA_FILE.exists():
    raise FileNotFoundError(f"Recipe data not found: {RECIPE_DATA_FILE}")

with open(RECIPE_DATA_FILE, 'r', encoding='utf-8') as f:
    recipes = json.load(f)

print(f"✓ Loaded {len(recipes):,} recipes")
print(f"\nSample recipe:")
print(json.dumps(recipes[0], indent=2)[:500] + "...")

Loading recipe data...
✓ Loaded 7,913 recipes

Sample recipe:
{
  "Unnamed: 0": 0,
  "recipe_title": "No-Bake Nut Cookies",
  "ingredients": [
    "1 c. firmly packed brown sugar",
    "1/2 c. evaporated milk",
    "1/2 tsp. vanilla",
    "1/2 c. broken nuts (pecans)",
    "2 Tbsp. butter or margarine",
    "3 1/2 c. bite size shredded rice biscuits"
  ],
  "instructions": [
    "In a heavy 2-quart saucepan, mix brown sugar, nuts, evaporated milk and butter or margarine.",
    "Stir over medium heat until mixture bubbles all over top.",
    "Boil and stir ...


In [16]:
# Format recipes for training
def format_recipe_for_training(recipe):
    """
    Convert recipe dict to training format
    Format: Instruction-following with clear structure
    """
    # Extract fields
    title = recipe.get('title', 'Unknown Recipe')
    ingredients = recipe.get('ingredients', [])
    directions = recipe.get('directions', [])
    
    # Format ingredients
    if isinstance(ingredients, list):
        ingredients_str = '\n'.join([f"- {ing}" for ing in ingredients[:15]])  # Limit to 15
    else:
        ingredients_str = str(ingredients)
    
    # Format instructions
    if isinstance(directions, list):
        instructions_str = '\n'.join([f"{i+1}. {step}" for i, step in enumerate(directions[:12])])  # Limit to 12
    else:
        instructions_str = str(directions)
    
    # Create training text (Llama 3.2 Instruct format)
    text = f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are an expert chef assistant. Generate clear, detailed recipes with proper formatting.<|eot_id|><|start_header_id|>user<|end_header_id|>

Generate a recipe titled "{title}".<|eot_id|><|start_header_id|>assistant<|end_header_id|>

# {title}

## Ingredients
{ingredients_str}

## Instructions
{instructions_str}<|eot_id|>"""
    
    return text

# Test formatting
sample_text = format_recipe_for_training(recipes[0])
print("Sample formatted text:")
print(sample_text[:800] + "...")

Sample formatted text:
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are an expert chef assistant. Generate clear, detailed recipes with proper formatting.<|eot_id|><|start_header_id|>user<|end_header_id|>

Generate a recipe titled "Unknown Recipe".<|eot_id|><|start_header_id|>assistant<|end_header_id|>

# Unknown Recipe

## Ingredients
- 1 c. firmly packed brown sugar
- 1/2 c. evaporated milk
- 1/2 tsp. vanilla
- 1/2 c. broken nuts (pecans)
- 2 Tbsp. butter or margarine
- 3 1/2 c. bite size shredded rice biscuits

## Instructions
<|eot_id|>...


In [17]:
# Prepare dataset
print("Preparing dataset...")

# Format all recipes
formatted_texts = [format_recipe_for_training(recipe) for recipe in recipes]

# Create HuggingFace dataset
dataset = Dataset.from_dict({"text": formatted_texts})

# Split into train/validation (90/10)
dataset = dataset.train_test_split(test_size=0.1, seed=42)

print(f"✓ Training samples: {len(dataset['train']):,}")
print(f"✓ Validation samples: {len(dataset['test']):,}")

Preparing dataset...
✓ Training samples: 7,121
✓ Validation samples: 792


## 4. Load Model with QLoRA

In [18]:
# QLoRA configuration
# 4-bit quantization for memory efficiency
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print("✓ QLoRA config created (4-bit quantization)")

✓ QLoRA config created (4-bit quantization)


In [19]:
try:
      tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-1B-Instruct")
      print("✅ 有存取權限！可以開始訓練")
      print(f"Vocab size: {len(tokenizer):,}")
except Exception as e:
      if "401" in str(e) or "gated" in str(e).lower():
          print("❌ 還沒有權限，請到以下網址申請：")
          print("https://huggingface.co/meta-llama/Llama-3.2-1B-Instruct")
      else:
          print(f"❌ 其他錯誤: {e}")

✅ 有存取權限！可以開始訓練
Vocab size: 128,256


In [20]:
# Load tokenizer
print("Loading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print(f"✓ Tokenizer loaded")
print(f"  Vocab size: {len(tokenizer):,}")

Loading tokenizer...
✓ Tokenizer loaded
  Vocab size: 128,256


In [21]:
# Load model with 4-bit quantization
print("Loading Llama 3.2 1B model...")
print("This may take 2-3 minutes...")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

# Prepare model for k-bit training
model = prepare_model_for_kbit_training(model)

print(f"✓ Model loaded")
print(f"  Parameters: {model.num_parameters() / 1e9:.2f}B")
print(f"  Memory footprint: {model.get_memory_footprint() / 1024**3:.2f} GB")

Loading Llama 3.2 1B model...
This may take 2-3 minutes...
✓ Model loaded
  Parameters: 1.24B
  Memory footprint: 1.43 GB


In [22]:
# LoRA configuration
# Only train ~1% of parameters for efficiency
peft_config = LoraConfig(
    r=8,                       # LoRA rank (reduced for 1B model)
    lora_alpha=16,             # LoRA alpha (2x rank)
    lora_dropout=0.05,         # Dropout
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[           # Which layers to apply LoRA
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
    ]
)

# Apply LoRA
model = get_peft_model(model, peft_config)

# Print trainable parameters
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
all_params = sum(p.numel() for p in model.parameters())
trainable_percent = 100 * trainable_params / all_params

print(f"✓ LoRA applied")
print(f"  Trainable params: {trainable_params:,} ({trainable_percent:.2f}%)")
print(f"  All params: {all_params:,}")

✓ LoRA applied
  Trainable params: 1,703,936 (0.23%)
  All params: 750,979,072


## 5. Training Configuration

In [28]:
# Training arguments
training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    
    # Training schedule
    num_train_epochs=3,                    # Number of epochs
    per_device_train_batch_size=1,         # Batch size (increase if you have more VRAM)
    gradient_accumulation_steps=4,         # Accumulate gradients (effective batch size = 4)
    
    # Optimization
    learning_rate=2e-4,
    warmup_steps=100,
    lr_scheduler_type="cosine",
    optim="adamw_torch",              # Memory-efficient optimizer
    
    # Logging and saving
    logging_steps=10,
    save_steps=500,
    save_total_limit=3,
    
    # Evaluation
    evaluation_strategy="steps",
    eval_steps=500,
    
    # Performance
    fp16=True,                             # Use mixed precision
    gradient_checkpointing=True,           # Save memory
    
    # Other
    report_to="none",                      # Disable wandb/tensorboard
    load_best_model_at_end=True,
)

print("✓ Training arguments configured")
print(f"  Effective batch size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"  Total training steps: ~{len(dataset['train']) * training_args.num_train_epochs // (training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps)}")

✓ Training arguments configured
  Effective batch size: 4
  Total training steps: ~5340


c:\Users\Champion\anaconda3\envs\pytorch_cuda\Lib\site-packages\transformers\training_args.py:1559: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


## 6. Initialize Trainer

In [29]:
# Initialize SFTTrainer (Supervised Fine-Tuning)
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    tokenizer=tokenizer,
    dataset_text_field="text",
    max_seq_length=512,  # Reduced for faster training on 1B model
)

print("✓ Trainer initialized")
print("\nReady to train!")

c:\Users\Champion\anaconda3\envs\pytorch_cuda\Lib\site-packages\huggingface_hub\utils\_deprecation.py:100: FutureWarning: Deprecated argument(s) used in '__init__': dataset_text_field, max_seq_length. Will not be supported from version '0.13.0'.

Deprecated positional argument(s) used in SFTTrainer, please use the SFTConfig to set these arguments instead.
  warnings.warn(message, FutureWarning)
c:\Users\Champion\anaconda3\envs\pytorch_cuda\Lib\site-packages\trl\trainer\sft_trainer.py:300: UserWarning: You passed a `max_seq_length` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(
c:\Users\Champion\anaconda3\envs\pytorch_cuda\Lib\site-packages\trl\trainer\sft_trainer.py:328: UserWarning: You passed a `dataset_text_field` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(


Map:   0%|          | 0/7121 [00:00<?, ? examples/s]

Map:   0%|          | 0/792 [00:00<?, ? examples/s]

✓ Trainer initialized

Ready to train!


## 7. Train Model

**Estimated time for Llama 3.2 1B:**
- RTX 3060 (6GB): ~1-1.5 hours
- RTX 3090 (24GB): ~45 minutes
- RTX 4090 (24GB): ~30 minutes

In [ ]:
# Start training
print("Starting training...\n")
print("This will take several hours. Monitor the loss below.")
print("Loss should decrease from ~2.0 to ~0.5-1.0\n")

trainer.train()

print("\n✓ Training completed!")

Starting training...

This will take several hours. Monitor the loss below.
Loss should decrease from ~2.0 to ~0.5-1.0



  0%|          | 0/5340 [00:00<?, ?it/s]

## 8. Save Model

In [33]:
# Test the fine-tuned model
print("Testing generation...\n")

# Merge LoRA weights (required for pipeline)
model = model.merge_and_unload()

# Test prompt
test_prompt = """<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are an expert chef assistant. Generate clear, detailed recipes with proper formatting.<|eot_id|><|start_header_id|>user<|end_header_id|>

Generate a recipe for Teriyaki Chicken.<|eot_id|><|start_header_id|>assistant<|end_header_id|>

"""

# Generate using model directly
inputs = tokenizer(test_prompt, return_tensors="pt").to(model.device)
outputs = model.generate(
    **inputs,
    max_new_tokens=512,
    temperature=0.7,
    top_p=0.9,
    do_sample=True,
    pad_token_id=tokenizer.eos_token_id
)

# Decode output
full_output = tokenizer.decode(outputs[0], skip_special_tokens=False)

# Extract only assistant's response
response = full_output.split("<|start_header_id|>assistant<|end_header_id|>")[-1]
response = response.split("<|eot_id|>")[0].strip()

print("Generated Recipe:")
print("=" * 80)
print(response)
print("=" * 80)

Testing generation...



c:\Users\Champion\anaconda3\envs\pytorch_cuda\Lib\site-packages\peft\tuners\lora\bnb.py:336: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(


Generated Recipe:
# Teriyaki Chicken Recipe

## Ingredients
- 1/2 c. soy sauce
- 1/2 c. sugar
- 1/4 c. vinegar
- 1/4 c. water
- 2 Tbsp. sesame oil
- 2 tsp. grated ginger
- 2 Tbsp. brown sugar
- 2 Tbsp. brown rice vinegar
- 2 Tbsp. cornstarch
- 2 Tbsp. vegetable oil
- 1 tsp. ground ginger
- 2 Tbsp. soy sauce
- 1 Tbsp. cornstarch
- 1/4 tsp. red pepper flakes
- 1/4 tsp. garlic powder
- 1/4 tsp. onion powder
- 1/4 tsp. salt
- 1/4 tsp. pepper
- 1 lb. chicken breast, cut into bite-sized pieces
- 1/4 c. sliced green onions
- 1/4 c. sliced mushrooms
- 1/4 c. sliced carrots
- 2 Tbsp. sliced green onions
- 1 Tbsp. sesame seed

## Instructions
1. In a small saucepan, combine soy sauce, sugar, vinegar, water, sesame oil, ginger, brown sugar, cornstarch, vegetable oil, garlic powder, onion powder, salt, pepper, and red pepper flakes. Bring to a boil, then reduce heat to low and simmer for 5 minutes.
2. In a small bowl, whisk together the remaining ingredients.
3. In a separate saucepan, combine the

In [34]:
# 測試：從食材生成食譜
def generate_recipe_from_ingredients(model, tokenizer, ingredients_str, cuisine=None):
    """
    從食材生成食譜（使用目前的微調模型）
    """
    cuisine_hint = f" ({cuisine} style)" if cuisine else ""
    
    prompt = f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are an expert chef assistant. Generate creative, detailed recipes from given ingredients.<|eot_id|><|start_header_id|>user<|end_header_id|>

Create a delicious recipe using these ingredients: {ingredients_str}{cuisine_hint}.<|eot_id|><|start_header_id|>assistant<|end_header_id|>

# """
    
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=512,
        temperature=0.8,  # 提高創意
        top_p=0.9,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
        repetition_penalty=1.1  # 避免重複
    )
    
    full_output = tokenizer.decode(outputs[0], skip_special_tokens=False)
    response = full_output.split("<|start_header_id|>assistant<|end_header_id|>")[-1]
    response = response.split("<|eot_id|>")[0].strip()
    
    return response

# 測試 1: 你的真實案例（照片檢測到的食材）
print("=" * 80)
print("測試 1: Chicken breast and Pork kidney")
print("=" * 80)
result1 = generate_recipe_from_ingredients(
    model, tokenizer,
    "chicken breast and pork kidney",
    cuisine="Asian"
)
print(result1)
print("\n")

# 測試 2: 簡單食材
print("=" * 80)
print("測試 2: Chicken breast, garlic, onion")
print("=" * 80)
result2 = generate_recipe_from_ingredients(
    model, tokenizer,
    "chicken breast, garlic, onion",
    cuisine="Mediterranean"
)
print(result2)
print("\n")

# 測試 3: 西式食材
print("=" * 80)
print("測試 3: Beef, potato, carrot")
print("=" * 80)
result3 = generate_recipe_from_ingredients(
    model, tokenizer,
    "beef, potato, carrot",
    cuisine="Western"
)
print(result3)

測試 1: Chicken breast and Pork kidney
# 3-Cheese Chicken Stuffed Pork Kidney Recipe

## Ingredients
- 1 large pork kidney, cleaned
- 1/2 lb. ground chicken
- 1 small onion, finely chopped
- 1 green pepper, finely chopped
- 2 cloves garlic, minced
- 2 tbsp. soy sauce
- 1 tsp. ginger powder
- 1 tsp. chili powder
- 1/4 c. brown sugar
- 1 Tbsp. cornstarch
- 1/4 c. water
- 8 oz. cream cheese
- 2 eggs
- 1/4 c. shredded Mozzarella cheese
- 1 Tbsp. butter or margarine
- salt and pepper to taste
- sliced scallions for garnish

## Instructions
1. Preheat oven to 375°F.
2. Clean the pork kidney and rinse with cold water.
3. Rinse off excess fat.
4. Cut the kidney into thin strips, about 1/4 inch thick. Reserve some meat in larger pieces.
5. In a medium bowl, combine ground chicken, chopped onion, chopped green pepper, grated ginger, soy sauce, chili powder, and garlic. Mix well.
6. Add the chopped parsley to the mixture and stir until combined.
7. In a separate bowl, mix together flour and cornsta

## 9. Test Generation

In [ ]:
# Test the fine-tuned model
print("Testing generation...\n")

# Create generation pipeline
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=512,
    temperature=0.7,
    top_p=0.9,
    do_sample=True,
)

# Test prompt
test_prompt = """<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are an expert chef assistant. Generate clear, detailed recipes with proper formatting.<|eot_id|><|start_header_id|>user<|end_header_id|>

Generate a recipe for Teriyaki Chicken.<|eot_id|><|start_header_id|>assistant<|end_header_id|>

"""

# Generate
output = pipe(test_prompt)[0]['generated_text']

# Print only the assistant's response
response = output.split("<|start_header_id|>assistant<|end_header_id|>")[-1]
response = response.split("<|eot_id|>")[0]

print("Generated Recipe:")
print("=" * 80)
print(response)
print("=" * 80)

## 10. Convert to GGUF (Optional)

To use the model with llama.cpp (faster inference):

```bash
# 1. Merge LoRA weights
python -m llama_cpp.server.merge_lora \
    --base_model meta-llama/Llama-3.2-1B-Instruct \
    --lora_model ./models/recipe_generation/llama3_1b_finetuned \
    --output_dir ./models/recipe_generation/llama3_1b_merged

# 2. Convert to GGUF
python convert.py ./models/recipe_generation/llama3_1b_merged

# 3. Quantize to Q5
./quantize ./models/recipe_generation/llama3_1b_merged/ggml-model-f16.gguf \
    ./models/recipe_generation/llama3_1b_merged/ggml-model-q5_k_m.gguf q5_k_m
```

## Summary

✓ **Model**: Llama 3.2 1B Instruct  
✓ **Method**: QLoRA (4-bit quantization)  
✓ **Training**: 3 epochs on recipe dataset  
✓ **Output**: Saved to `models/recipe_generation/llama3_1b_finetuned`  
✓ **GPU Requirement**: 6GB VRAM (RTX 3060 or better)

**Next steps:**
1. Load this model in your recipe generation pipeline
2. Compare quality with GPT-2
3. Optionally convert to GGUF for faster inference

**Expected improvements over GPT-2:**
- Instructions quality: +20-30%
- No gibberish or non-Latin characters
- Better instruction following
- Reduced need for filtering
- More consistent recipe structure